# Análisis Exporatorio de los datos de un año entero

### 0. Importación de Librerías y Configuración

In [2]:
import pandas as pd
import glob
import os
import seaborn as sns
import matplotlib.pyplot as plt

# Configuración de estilo
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

### 1. Carga y Configuración de Datos

In [3]:
BASE_DIR = "../../data/processed/tlc_clean"
ANIO = "2025" 
SAMPLE_RATE = 0.10  # 10% de cada mes

In [6]:
# --- FUNCIÓN DE CARGA ---

def carga_datos_todosmeses():
    lista_muestras = []
    
    # Iteramos por los dos tipos de servicio que menciona el contexto: Yellow y VTC
    for tipo in ['yellow', 'fhvhv']:
        ruta_busqueda = os.path.join(BASE_DIR, tipo, ANIO, "*.parquet")
        archivos_meses = sorted(glob.glob(ruta_busqueda))
        
        if not archivos_meses:
            print(f"No se encontraron archivos para {tipo} en {ANIO}")
            continue
            
        print(f"Procesando {len(archivos_meses)} meses de {tipo}...")
        
        for archivo in archivos_meses:
            # 1. Leer el archivo del mes completo
            df_mes = pd.read_parquet(archivo)
            
            # 2. Extraer el 10% de forma aleatoria pero reproducible
            df_sample = df_mes.sample(frac=SAMPLE_RATE, random_state=42)
            
            # 3. Añadir etiquetas para no perder la información al unir
            df_sample['tipo_vehiculo'] = 'Yellow Taxi' if tipo == 'yellow' else 'VTC'
            
            # Aseguramos que la fecha sea datetime y extraemos el mes
            df_sample['fecha_inicio'] = pd.to_datetime(df_sample['fecha_inicio'])
            df_sample['mes_num'] = df_sample['fecha_inicio'].dt.month
            
            lista_muestras.append(df_sample)
            print(f"Mes {df_sample['mes_num'].iloc[0]} cargado ({len(df_sample):,} registros)")

    # 4. Concatenar todas las muestras en un único DataFrame
    df_final = pd.concat(lista_muestras, ignore_index=True)
    
    # 5. Ordenar por mes y tipo para que las gráficas salgan perfectas
    df_final = df_final.sort_values(by=['mes_num', 'tipo_vehiculo'])
    
    # Resetear el índice para que sea limpio
    df_final = df_final.reset_index(drop=True)
    
    print(f"\nProceso finalizado. Dataset creado con {len(df_final):,} filas totales.")
    return df_final

In [7]:
# Ejecución
df_anual = carga_datos_todosmeses()
df_anual.head()
# Guardar el resultado para no tener que repetir el proceso cada vez que abras el notebook
# df_anual.to_parquet("df_ny_2025_sample_10pct.parquet")


Procesando 2 meses de yellow...
Mes 1 cargado (280,150 registros)
Mes 2 cargado (264,376 registros)
Procesando 2 meses de fhvhv...
Mes 1 cargado (1,460,435 registros)
Mes 2 cargado (1,368,289 registros)

Proceso finalizado. Dataset creado con 3,373,250 filas totales.


,fecha_inicio,fecha_fin,origen_id,destino_id,num_pasajeros,distancia,duracion_min,velocidad_mph,tipo_vehiculo,precio_base,...,destino_barrio,temp_c,precipitation,viento_kmh,lluvia,nieve,es_festivo,hay_evento,mes_num,espera_min
0,2025-01-27 21:49:36,2025-01-27 22:09:03,231,163,<NA>,7.26,19.450000,22.395888,VTC,57.610001,...,Manhattan,0.3,0.0,18.2,0.0,0.0,0,0,1,3.116667
1,2025-01-26 06:53:37,2025-01-26 06:58:36,42,247,<NA>,0.95,4.983333,11.438127,VTC,12.340000,...,Bronx,-1.9,0.0,13.4,0.0,0.0,0,0,1,6.700000
2,2025-01-21 13:49:15,2025-01-21 14:18:41,68,232,<NA>,5.12,29.433333,10.437146,VTC,35.830002,...,Manhattan,-8.6,0.0,6.1,0.0,0.0,0,0,1,4.333333
3,2025-01-09 20:40:10,2025-01-09 21:03:25,130,255,<NA>,11.22,23.250000,28.954839,VTC,29.510000,...,Brooklyn,-2.2,0.0,22.7,0.0,0.0,0,0,1,4.700000
4,2025-01-10 22:39:44,2025-01-10 23:00:45,144,112,<NA>,4.30,21.016667,12.275972,VTC,31.600000,...,Brooklyn,-0.3,0.0,3.6,0.0,0.0,0,0,1,2.550000
